<a href="https://colab.research.google.com/github/irhamadm/Dummy-Data-Mining/blob/main/Forecasting_Diabetes_APP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import streamlit as st
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import accuracy_score
import pickle
import os

st.set_page_config(
    page_title="Aplikasi Prediksi Diabetes",
    page_icon="🩸",
    layout="centered",
    initial_sidebar_state="auto"
)

csv_file_path = r'C:\Users\Irham\OneDrive\Dokumen\Streamlit\diabetes.csv'

model_filename = 'diabetes_model.sav'
scaler_filename = 'scaler.sav'

def train_and_save_model_logic():
    if not os.path.exists(csv_file_path):
        return None, None, f"Error: File dataset '{csv_file_path}' tidak ditemukan. Mohon pastikan 'diabetes.csv' ada di lokasi tersebut. Aplikasi tidak dapat berjalan tanpa data pelatihan."

    try:
        diabetes_dataset = pd.read_csv(csv_file_path)
    except Exception as e:
        return None, None, f"Error saat membaca file CSV: {e}. Pastikan file tidak rusak."

    X = diabetes_dataset.drop(columns = 'Outcome', axis=1)
    Y = diabetes_dataset['Outcome']

    scaler = StandardScaler()
    scaler.fit(x)
    X_scaled = scaler.transform(x)

    X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y, test_size = 0.2, stratify=Y, random_state=2)

    classifier = svm.SVC(kernel='linear')
    classifier.fit(X_train, Y_train)

    pickle.dump(classifier, open(model_filename, 'wb'))
    pickle.dump(scaler, open(scaler_filename, 'wb'))

    return classifier, scaler, None # Mengembalikan model, scaler, dan tanpa error

@st.cache_resource
def get_model_and_scaler_cached():
    model = None
    scaler = None
    status_message = ""

    if os.path.exists(model_filename) and os.path.exists(scaler_filename):
        try:
            model = pickle.load(open(model_filename, 'rb'))
            scaler = pickle.load(open(scaler_filename, 'rb'))
            status_message = "Model dan Scaler berhasil dimuat."
        except Exception as e:
            status_message = f"Terjadi kesalahan saat memuat model/scaler yang sudah ada: {e}. Mencoba melatih ulang..."
            model, scaler, error_train = train_and_save_model_logic()
            if error_train:
                status_message = error_train # Ganti pesan status dengan error pelatihan
    else:
        status_message = "Model dan Scaler belum ditemukan. Melatih model dari awal..."
        model, scaler, error_train = train_and_save_model_logic()
        if error_train:
            status_message = error_train # Ganti pesan status dengan error pelatihan

    return model, scaler, status_message

loaded_model, loaded_scaler, initial_load_status = get_model_and_scaler_cached()

if initial_load_status:
    if "Error" in initial_load_status:
        st.error(initial_load_status)
    elif "berhasil dimuat" in initial_load_status or "berhasil dilatih" in initial_load_status:
        st.sidebar.success(initial_load_status)
    else:
        st.sidebar.info(initial_load_status)

st.title("🩺 Sistem Prediksi Resiko Diabetes")
st.markdown("Aplikasi ini memprediksi kemungkinan seseorang terkena diabetes berdasarkan input parameter medis.")
st.markdown("---")

if loaded_model is None or loaded_scaler is None:
    st.warning("Model atau Scaler tidak tersedia. Mohon periksa pesan error di atas dan pastikan dataset Anda ada.")
    st.stop() # Hentikan eksekusi lebih lanjut jika model/scaler tidak berhasil dimuat/dilatih


st.header("Masukkan Data Pasien:")

col1, col2, col3 = st.columns(3)

with col1:
    pregnancies = st.number_input("Jumlah Kehamilan", min_value=0, max_value=17, value=1, help="Jumlah kehamilan.")
    glucose = st.number_input("Kadar Glukosa (mg/dL)", min_value=0, max_value=200, value=120, help="Konsentrasi glukosa plasma 2 jam.")
    blood_pressure = st.number_input("Tekanan Darah (mmHg)", min_value=0, max_value=122, value=70, help="Tekanan darah diastolik.")

with col2:
    skin_thickness = st.number_input("Ketebalan Kulit (mm)", min_value=0, max_value=99, value=20, help="Ketebalan lipatan kulit trisep.")
    insulin = st.number_input("Kadar Insulin (muU/ml)", min_value=0, max_value=846, value=79, help="Kadar insulin serum 2 jam.")
    bmi = st.number_input("Indeks Massa Tubuh (BMI)", min_value=0.0, max_value=185.9, value=25.0, help="BMI (berat badan dalam kg / (tinggi dalam m)^2).")

with col3:
    diabetes_pedigree_function = st.number_input("Fungsi Pedigree Diabetes", min_value=0.00, max_value=2.50, value=0.4, format="%.3f", help="Mengukur riwayat diabetes dalam keluarga.")
    age = st.number_input("Usia (tahun)", min_value=1, max_value=100, value=30, help="Usia pasien.")

st.markdown("---")
if st.button("Tampilkan Hasil Analisis"):
    try:
        input_data = np.array([
            pregnancies, glucose, blood_pressure, skin_thickness,
            insulin, bmi, diabetes_pedigree_function, age
        ]).reshape(1, -1)

        std_data = loaded_scaler.transform(input_data)
        prediction = loaded_model.predict(std_data)

        st.subheader("Hasil Analisis:")
        if prediction[0] == 0:
            st.success("✅ *Pasien TIDAK beresiko diabetes.*")
            st.write("Berdasarkan data yang Anda masukkan, model memprediksi bahwa pasien tidak memiliki diabetes.")
        else:
            st.warning("⚠ *Pasien BERESIKO diabetes.*")
            st.write("Berdasarkan data yang Anda masukkan, model memprediksi bahwa pasien memiliki diabetes.")

        st.markdown("---")
        st.info("Catatan Penting: Prediksi ini hanya berdasarkan model machine learning dan data yang dilatih. Ini BUKAN diagnosis medis yang akurat. Selalu konsultasikan dengan profesional medis untuk diagnosis dan saran kesehatan.")

    except Exception as e:
        st.error(f"Terjadi kesalahan saat melakukan prediksi. Mohon periksa input Anda: {e}")
        st.info("Mohon hubungi pengembang jika masalah berlanjut.")

st.markdown("---")
st.caption("Aplikasi ini dibuat dengan Streamlit dan Scikit-learn untuk tujuan demonstrasi.")